# Starbucks Rewards - EDA inicial

Objetivo desta etapa: ler os JSONs em `data/raw` e validar estrutura mínima antes de qualquer análise preditiva ou causal.

Escopo desta EDA:
- shape, colunas, tipos e primeiras linhas;
- nulos e duplicados;
- eventos do `transcript`;
- inspeção do campo aninhado `value`.

Não há modelagem nesta etapa.

## 1. Setup e leitura dos dados

Os arquivos são JSON Lines: cada linha contém um registro JSON independente.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

files = {
    "portfolio": RAW_DIR / "portfolio.json",
    "profile": RAW_DIR / "profile.json",
    "transcript": RAW_DIR / "transcript.json",
}

for name, path in files.items():
    print(f"{name}: {path} | exists={path.exists()} | size_mb={path.stat().st_size / 1024**2:.2f}")

In [ ]:
dfs = {
    name: pd.read_json(path, lines=True)
    for name, path in files.items()
}

portfolio = dfs["portfolio"]
profile = dfs["profile"]
transcript = dfs["transcript"]

## 2. Shape, colunas, tipos e head

In [ ]:
for name, df in dfs.items():
    print("=" * 80)
    print(name.upper())
    print(f"shape: {df.shape}")
    print("\ncolunas:")
    print(list(df.columns))
    print("\ntipos:")
    print(df.dtypes)
    display(df.head())

## 3. Nulos e duplicados

Duplicados abaixo consideram a linha inteira. Unicidade de chaves será validada em etapa posterior, depois de confirmar a granularidade esperada.

In [ ]:
import json


def make_hashable_for_duplicate_check(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, sort_keys=True)
    return value


def count_full_row_duplicates(df):
    comparable_df = df.map(make_hashable_for_duplicate_check)
    return int(comparable_df.duplicated().sum())


quality_rows = []

for name, df in dfs.items():
    quality_rows.append(
        {
            "tabela": name,
            "linhas": len(df),
            "colunas": df.shape[1],
            "duplicados_linha_inteira": count_full_row_duplicates(df),
        }
    )

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)

In [ ]:
for name, df in dfs.items():
    nulls = (
        df.isna()
        .sum()
        .rename("nulos")
        .to_frame()
    )
    nulls["pct_nulos"] = (nulls["nulos"] / len(df)).round(4)

    print("=" * 80)
    print(name.upper())
    display(nulls.sort_values("nulos", ascending=False))

## 4. Eventos do transcript

Esta etapa apenas descreve os eventos disponíveis e sua distribuição temporal. Ainda não atribui transações a ofertas nem define tratamento, controle ou outcome.

In [ ]:
event_summary = (
    transcript["event"]
    .value_counts(dropna=False)
    .rename_axis("event")
    .reset_index(name="registros")
)
event_summary["pct"] = (event_summary["registros"] / len(transcript)).round(4)

display(event_summary)

In [ ]:
time_summary = (
    transcript
    .groupby("event", dropna=False)["time"]
    .agg(registros="count", min_time="min", max_time="max", median_time="median")
    .reset_index()
    .sort_values("event")
)

display(time_summary)

## 5. Inspeção do campo `value`

O campo `value` é aninhado e muda conforme o tipo de evento. Esta inspeção identifica suas chaves sem assumir ainda regra de atribuição entre oferta e transação.

In [ ]:
def value_keys(value):
    if isinstance(value, dict):
        return tuple(sorted(value.keys()))
    return tuple()


value_key_summary = (
    transcript
    .assign(value_keys=transcript["value"].map(value_keys))
    .groupby(["event", "value_keys"], dropna=False)
    .size()
    .reset_index(name="registros")
    .sort_values(["event", "registros"], ascending=[True, False])
)

display(value_key_summary)

In [ ]:
value_expanded = pd.json_normalize(transcript["value"])
value_expanded.columns = [col.replace(" ", "_") for col in value_expanded.columns]

transcript_value_preview = pd.concat(
    [transcript[["person", "event", "time"]], value_expanded],
    axis=1,
)

print("colunas extraidas de value:")
print(list(value_expanded.columns))

display(transcript_value_preview.head(10))

### 5.1. Correção do campo `value`

Nesta etapa, o campo `value` é aberto em colunas analíticas simples. As chaves `offer id` e `offer_id` são unificadas em uma única coluna `offer_id`.

Não há join, remoção de duplicados ou modelagem.

In [ ]:
value_expanded_raw = pd.json_normalize(transcript["value"])

offer_id = pd.Series(pd.NA, index=transcript.index)
if "offer id" in value_expanded_raw.columns:
    offer_id = offer_id.combine_first(value_expanded_raw["offer id"])
if "offer_id" in value_expanded_raw.columns:
    offer_id = offer_id.combine_first(value_expanded_raw["offer_id"])

transcript_clean = transcript[["person", "event", "time"]].copy()
transcript_clean["offer_id"] = offer_id
transcript_clean["amount"] = value_expanded_raw.get("amount")
transcript_clean["reward_event"] = value_expanded_raw.get("reward")

transcript_clean = transcript_clean[
    ["person", "event", "time", "offer_id", "amount", "reward_event"]
]

display(transcript_clean.head(10))

In [ ]:
value_validation = (
    transcript_clean
    .groupby("event", dropna=False)
    .agg(
        registros=("event", "size"),
        offer_id_preenchido=("offer_id", "count"),
        amount_preenchido=("amount", "count"),
        reward_event_preenchido=("reward_event", "count"),
    )
    .reset_index()
)

value_validation["pct_offer_id"] = (
    value_validation["offer_id_preenchido"] / value_validation["registros"]
).round(4)
value_validation["pct_amount"] = (
    value_validation["amount_preenchido"] / value_validation["registros"]
).round(4)
value_validation["pct_reward_event"] = (
    value_validation["reward_event_preenchido"] / value_validation["registros"]
).round(4)

display(value_validation)

In [ ]:
expected_value_rules = pd.DataFrame(
    [
        {
            "validacao": "transaction tem amount",
            "ok": transcript_clean.loc[
                transcript_clean["event"].eq("transaction"), "amount"
            ].notna().all(),
        },
        {
            "validacao": "eventos de oferta tem offer_id",
            "ok": transcript_clean.loc[
                transcript_clean["event"].isin(
                    ["offer received", "offer viewed", "offer completed"]
                ),
                "offer_id",
            ].notna().all(),
        },
        {
            "validacao": "offer completed tem reward_event",
            "ok": transcript_clean.loc[
                transcript_clean["event"].eq("offer completed"), "reward_event"
            ].notna().all(),
        },
    ]
)

display(expected_value_rules)

In [ ]:
for event_name in transcript["event"].dropna().sort_values().unique():
    print("=" * 80)
    print(event_name)
    display(
        transcript.loc[transcript["event"] == event_name, ["person", "event", "value", "time"]]
        .head(5)
    )

## Etapa 3 - Entidades e granularidade

Esta etapa documenta as entidades e a granularidade do dataset antes de qualquer join, construção de ABT ou modelagem.

### Entidades principais

- cliente: pessoa cadastrada no programa de recompensas. A chave candidata é profile.id, que se conecta aos eventos por 	ranscript_clean.person.
- oferta: campanha ou incentivo disponível no catálogo. A chave candidata é portfolio.id, que se conecta aos eventos por 	ranscript_clean.offer_id.
- evento: cada linha do 	ranscript ou 	ranscript_clean, representando uma ocorrência temporal para um cliente, com person, event, 	ime e campos derivados de alue.
- 	ransação: evento do tipo 	ransaction, representando comportamento financeiro. Deve ter mount preenchido e não possui offer_id direto no próprio evento.

### Chaves de ligação candidatas

- Cliente: profile.id = 	ranscript_clean.person.
- Oferta: portfolio.id = 	ranscript_clean.offer_id.
- Tempo: 	ranscript_clean.time, ainda dependente de validação da unidade temporal.

### Melhor unidade de análise

Para a EDA inicial, a melhor unidade de análise continua sendo o evento, pois ainda estamos validando estrutura, chaves, datas relativas e o significado do campo alue.

Para uma análise causal ou de uplift futura, a unidade mais defensável tende a ser cliente-oferta recebida: uma linha por cliente, oferta e momento de recebimento (offer received). Essa unidade ainda não deve ser construída nesta etapa, porque exigiria regras temporais, atribuição de transações a janelas de oferta e joins entre tabelas.

### Cuidados causais

- offer received é o candidato natural ao momento de exposição ou tratamento.
- offer viewed ocorre após o recebimento da oferta e pode ser um evento pós-tratamento.
- offer completed também ocorre após a oferta e pode estar no caminho causal entre tratamento e outcome.
- Portanto, offer viewed e offer completed não devem ser usados como features pré-tratamento em modelos ou análises causais.


## 6. Observações para validação manual antes de avançar

- Confirmar a unidade do campo `time` e se ela é compatível com `duration` em `portfolio`.
- Confirmar se `profile.age == 118` representa idade real ou valor sentinela para cadastro incompleto.
- Confirmar se `value.offer id` e `value.offer_id` representam a mesma chave de oferta em eventos diferentes.
- Confirmar se existe grupo controle real ou se a ausência de oferta será apenas comparação observacional.
- Não definir tratamento, controle, outcome ou janela causal antes dessas validações.